**Part 2 · LangChain** · step 7 of 25  
[← 06 Prompt templates & chains](06_prompt_templates_and_chains.ipynb) · [🏠 Roadmap](../README.md) · [08 Conversation history →](08_conversation_history.ipynb)

# 07 · Structured Output

Instead of free text, make the model return data that matches a **schema** —
perfect for extraction, forms, APIs and anything your code needs to parse.

`model.with_structured_output(Schema)` returns a new runnable whose `invoke()` gives you a
**Pydantic object**, not an `AIMessage`. `Field(description=...)` text is sent to the model,
so describe each field clearly. (Pydantic basics: notebook 03.)

In [1]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-4.1")

## Example 1 — extracting movie details

In [3]:
from pydantic import BaseModel, Field


class Movie(BaseModel):
    """Details about a movie."""

    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie's rating out of 10")


model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [4]:
# It is a real Pydantic object: attribute access, validation, .model_dump() ...
print(response.title, response.year)
response.model_dump()

Inception 2010


{'title': 'Inception',
 'year': 2010,
 'director': 'Christopher Nolan',
 'rating': 8.8}

## Example 2 — extracting facts from free text

In [5]:
class Person(BaseModel):
    """A person mentioned in the text."""

    name: str
    age: int


extractor = init_chat_model("gpt-4o").with_structured_output(Person)
person = extractor.invoke("My name is Ujjawal and I am 27 years old")
person.name, person.age

('Ujjawal', 27)

## Where can I ask for structured output?

| Goal | Supported? | How |
|---|---|---|
| `init_chat_model(..., response_format=...)` | ❌ No | Not a supported argument |
| Structured Pydantic output from a model | ✅ Yes | `model.with_structured_output(MySchema)` |
| Structured output from an agent | ✅ Yes | `create_agent(response_format=MySchema)` → `result["structured_response"]` |
| Provider JSON mode | ✅ Sometimes | `model.bind(response_format=...)` |

---
✅ **Done with 07 Structured output.** Next: [08 Conversation history →](08_conversation_history.ipynb)  
[🏠 Roadmap](../README.md)